# Batch correction with scPoli (conditional VAE, scArches)

scPoli (De Donno et al., *Nat Methods* 2023) is a conditional VAE from the scArches ecosystem that learns one or more per-condition prototypes in the latent space. When given cell-type labels (`cell_type_keys`), the prototypes anchor each cell type across batches, which both improves batch removal and enables prototype-based label transfer to unseen (query) datasets.

This is one of the **omicverse batch-correction zoo** tutorials. For an overview of all backends, the recommendation tree, and the unified `_BATCH_OBSM` schema, see [batch/index](../index.md). For the side-by-side comparison of all backends on the NeurIPS 2021 multi-batch benchmark, see [t_single_batch](../t_single_batch.ipynb).

Optional dependency: `pip install scarches`.

scPoli takes `batch_key` as its `condition_keys` argument internally; the omicverse wrapper handles that. Pass `cell_type_keys=` (str or list) to enable prototype learning — without it scPoli still runs but you lose the label-transfer use case.


## Load a multi-batch dataset

We use the same toy multi-batch AnnData as the other zoo tutorials — three NeurIPS 2021 batches concatenated. Replace with your own dataset by changing `adata` and `batch_key` below.

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np

# Replace these URLs with your own dataset; the three are
# the same multi-batch dataset used by t_single_batch.
adata1 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932005',
    filename='neurips2021_s1d3.h5ad',
)
adata2 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932008',
    filename='neurips2021_s2d1.h5ad',
)
adata3 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932011',
    filename='neurips2021_s3d7.h5ad',
)
adata = sc.concat([adata1, adata2, adata3], merge='same')
adata.obs['batch'] = adata.obs['batch'].astype('category')
adata

## Preprocess + PCA (shared across all backends)

Every backend in the zoo starts from the same QC'd, log-normalised AnnData with `scaled|original|X_pca` in obsm. Read [t_single_batch](../t_single_batch.ipynb) for the full discussion of these steps.

In [ ]:
adata = ov.pp.qc(adata,
                 tresh={'mito_perc': 0.2, 'nUMIs': 500,
                        'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson',
                         n_HVGs=2000, batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=50)

## Run `ov.single.batch_correction(methods='scPoli')`

The wrapper routes method-specific kwargs to the right destination — for scvi-tools backends this includes splitting between `__init__` (architecture) and `.train()` (optimisation). See the **Key parameters** section below.

In [ ]:
model = ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='scPoli',
    # Recommended: enables prototype learning per cell type.
    cell_type_keys='celltype',
    # scPoli.__init__ params:
    embedding_dims=5, recon_loss='nb',
    # scPoli.train params:
    n_epochs=50, pretraining_epochs=40,
)
model

## Visualise the corrected embedding

Every backend writes its corrected representation to a stable obsm key — for **scPoli (conditional VAE, scArches)** it is `adata.obsm['X_scPoli']`. We project it via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_scpoli'] = ov.utils.mde(adata.obsm['X_scPoli'])
ov.pl.embedding(
    adata,
    basis='X_mde_scpoli',
    color=['batch'],
    frameon='small',
    title='scPoli (conditional VAE, scArches) — coloured by batch',
)

## Key parameters

**Optional but recommended**:
- `cell_type_keys` — obs column(s) with cell-type labels for prototype learning.

**Architecture** (routed to `scPoli.__init__`):
- `embedding_dims` — condition-embedding dim (default 10).
- `recon_loss` — `'nb'` / `'zinb'` / `'mse'`.
- `latent_dim` — latent dimension.

**Optimisation** (routed to `scPoli.train`):
- `n_epochs`, `pretraining_epochs`, `eta`, `early_stopping_kwargs`.


## Related tutorials

- `scANVI` — alternative semi-supervised approach.
- `scVI` — when you don't need prototype learning / label transfer.

For a side-by-side comparison of every backend on the same benchmark + scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).